# 03 — LoRA supervised fine-tuning (SFT)

We adapt the instruction-tuned Gemma 4 text decoder with LoRA. The base weights
remain frozen; small low-rank adapters learn to produce:

```
<reasoning>causal chain</reasoning>
<evidence>observation quoted from the task</evidence>
<answer>mechanistic conclusion</answer>
```

Device selection is automatic: **CUDA → MPS → CPU**. Gemma 4 E4B is a serious
model; MPS and CPU paths are functional fallbacks but may be impractically slow
on a laptop. Set `MODEL_ID` to a smaller chat model for an in-class laptop run.

## Prerequisites

Run notebooks 01–02 first. Accept the Gemma license on Hugging Face and authenticate:

```bash
hf auth login
```

Every model, LoRA, training, checkpoint, and Hub setting is defined in the
configuration cell below.

## Configuration

Change values here—do not create additional environment variables. Hugging Face
uses cached login credentials. The optional `HF_TOKEN` line may be uncommented
after exporting a token, but never paste a token into the notebook.

In [ ]:
import os

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_from_disk
from IPython.display import JSON, Markdown, display
from trl import SFTConfig, SFTTrainer

from science_course.devices import clear_device_cache, detect_runtime
from science_course.hub import require_hf_namespace
from science_course.modeling import (
    generate_text,
    load_causal_lm,
    load_tokenizer,
    render_prompt,
    render_sft_completion,
    teaching_lora_config,
)
from science_course.versions import require_training_stack

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Runtime and artifact locations
ENABLE_MPS_FALLBACK = True
if ENABLE_MPS_FALLBACK:
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
MODEL_ID = "google/gemma-4-E4B-it"
SFT_DATA = ROOT / "data" / "processed" / "sft"
OUTPUT_DIR = ROOT / "artifacts" / "gemma4-mechanism-sft"
RESUME_FROM_CHECKPOINT = None

# LoRA
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

# SFT
NUM_TRAIN_EPOCHS = 3
MAX_STEPS = -1
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQUENCE_LENGTH = 1_024
WARMUP_RATIO = 0.05
EVAL_STEPS = 25
SAVE_STEPS = 25
LOGGING_STEPS = 5
RANDOM_SEED = 17

# Hugging Face publication: every saved checkpoint plus the final adapter
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
SFT_HF_REPO = "lamm-mit/scientific-sft-grpo-sft"
PUSH_TO_HUB = True
HUB_PRIVATE_REPO = False
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

versions = require_training_stack()
runtime = detect_runtime()
if PUSH_TO_HUB:
    require_hf_namespace(SFT_HF_REPO, token=HF_TOKEN)

display(
    JSON(
        {
            "runtime": runtime.as_dict(),
            "versions": versions,
            "model": MODEL_ID,
            "hub_repo": SFT_HF_REPO,
        }
    )
)

## 1. Load and render the prompt-completion data

The tokenizer's own chat template formats the prompt. We disable any model-specific
thinking mode when the template supports that option. SFT loss is computed on the
reference completion, not the prompt.

In [ ]:
if not SFT_DATA.exists():
    raise RuntimeError("Run notebook 01 to create the SFT dataset.")
sft = load_from_disk(SFT_DATA)
if len(sft["train"]) == 0 or len(sft["validation"]) == 0:
    raise RuntimeError("Both SFT train and validation splits must be non-empty.")

tokenizer = load_tokenizer(MODEL_ID, token=HF_TOKEN)

def render_row(row):
    return {
        "prompt": render_prompt(tokenizer, row["prompt"]),
        "completion": render_sft_completion(tokenizer, row["completion"]),
    }

rendered = sft.map(
    render_row,
    remove_columns=sft["train"].column_names,
    desc="Apply the Gemma chat template",
)
display(JSON(rendered["train"][0]))

In [ ]:
lengths = pd.DataFrame(
    {
        split_name: pd.Series([
            len(tokenizer(item["prompt"] + item["completion"]).input_ids)
            for item in split_data
        ])
        for split_name, split_data in rendered.items()
    }
)
lengths.plot.hist(bins=20, alpha=0.65, figsize=(9, 4.5))
plt.axvline(
    MAX_SEQUENCE_LENGTH,
    color="crimson",
    linestyle="--",
    label="training max_length",
)
plt.xlabel("tokens")
plt.title("SFT sequence-length audit")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Load Gemma and attach LoRA

`AutoModelForCausalLM` loads only the text decoder. We do not quantize because the
same notebook must work on CUDA, Apple MPS, and CPU. LoRA targets linear layers;
only adapter parameters are optimized.

In [ ]:
clear_device_cache(runtime)
model = load_causal_lm(MODEL_ID, runtime, token=HF_TOKEN)
lora = teaching_lora_config(
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
)

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_length=MAX_SEQUENCE_LENGTH,
    completion_only_loss=True,
    loss_type="nll",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    report_to="none",
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=SFT_HF_REPO,
    hub_strategy="all_checkpoints",
    hub_private_repo=HUB_PRIVATE_REPO,
    hub_token=HF_TOKEN,
    hub_always_push=True,
    bf16=runtime.trainer_bf16,
    fp16=runtime.trainer_fp16,
    use_cpu=runtime.use_cpu,
    seed=RANDOM_SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=rendered["train"],
    eval_dataset=rendered["validation"],
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

## 3. Train

The progress bar reports optimization loss; periodic validation measures whether
held-out reference responses are also becoming more likely. Change `MAX_STEPS`
in the configuration cell for a bounded classroom run.

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
if PUSH_TO_HUB:
    hub_result = trainer.push_to_hub(
        commit_message="Complete mechanism SFT training"
    )
    print(f"Published final adapter and all checkpoints: {hub_result}")
display(JSON(train_result.metrics))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
if "loss" in history:
    history.dropna(subset=["loss"]).plot(
        x="step", y="loss", ax=axes[0], color="#315c8c", legend=False
    )
axes[0].set_title("Completion-only SFT loss")
if "eval_loss" in history:
    history.dropna(subset=["eval_loss"]).plot(
        x="step", y="eval_loss", ax=axes[1], color="#d97732", legend=False
    )
axes[1].set_title("Held-out SFT loss")
plt.tight_layout()
plt.show()

## 4. Use the adapter on a new task

This is the real deployment interface: give the fine-tuned model a new
self-contained mechanism task. No original paper text or reference answer is
required at inference time.

In [ ]:
new_task = (
    "A hydrogel contains polymer chains joined by reversible host–guest "
    "interactions. Mechanical strain separates some pairs, allowing local chain "
    "rearrangement. After the strain is removed, compatible host and guest groups "
    "associate again. Explain how these events allow the gel to recover after damage."
)
new_messages = [
    {
        "role": "system",
        "content": (
            "Solve the self-contained scientific mechanism task. Explain the causal "
            "chain, quote the most relevant observation already included in the task, "
            "and give a concise answer."
        ),
    },
    {
        "role": "user",
        "content": (
            f"SCIENTIFIC MECHANISM TASK\n{new_task}\n\n"
            "Respond with <reasoning>, <evidence>, and <answer> tags."
        ),
    },
]
prompt = render_prompt(tokenizer, new_messages)
response = generate_text(trainer.model, tokenizer, prompt, runtime)
display(Markdown(f"```text\n{response}\n```"))

## Result

`artifacts/gemma4-mechanism-sft/` contains a small LoRA adapter plus tokenizer
metadata—not a second copy of all Gemma weights. The final adapter and every saved
checkpoint are also published to `lamm-mit/scientific-sft-grpo-sft`. Notebook 04
loads that adapter as the starting policy and improves it with mechanism-sensitive
GRPO rewards.